In [1]:
import radiate as rd
import polars as pl

rd.random.seed(67123)

### Create a fake dataset 

This dataset is a simple polynomial regression problem with a small it of noise

In [2]:
def compute(x: float) -> float:
    return 4.0 * x**3 - 3.0 * x**2 + x


inputs = []
answers = []

input = -1.0
for _ in range(-10, 10):
    input += 0.1
    inputs.append([input])
    answers.append([compute(input)])

In [3]:
target_species = 4

distance_signal_mean = rd.Expr.select("species.distance").rolling(25).mean()
species_count_mean = rd.Expr.select("species.count").rolling(25).mean()

### Create the engine

This engine evolves `rd.Graph`s to fit the polynomial regression problem. The vertices of the graph are only allowed to use addition, multiplication, and constants. The engine also tracks the species count and species threshold as metrics, and uses the NEAT distance as a diversity metric.

In [4]:
collector = rd.MetricCollector()

engine = (
    rd.Engine.graph(
        shape=(1, 1),
        vertex=[rd.Op.sub(), rd.Op.mul(), rd.Op.sum()],
        edge=rd.Op.weight(),
        output=rd.Op.sum(),
    )
    .regression(inputs, answers, loss=rd.MSE)
    .subscribe(collector)
    .diversity(
        rd.Dist.neat(),
        # threshold=species_threshold,
        target=target_species,
    )
    .metrics(
        distance_signal_mean=distance_signal_mean,
        species_count_mean=species_count_mean,
    )
    .alter(
        rd.Cross.graph(0.05, 0.5),
        rd.Mutate.op(0.07, 0.05),
        rd.Mutate.graph(0.1, 0.1, False),
    )
    .limit(rd.Limit.score(0.001), rd.Limit.generations(1000))
)

result = engine.run(log=True)

2026-11-14T19:12:37.265900Z  INFO State Change: PreStart -> Running
2026-11-14T19:12:37.266682Z  INFO Epoch 1    | Score:   2.0038 | Time: 314.00µs
2026-11-14T19:12:37.266821Z  INFO Epoch 2    | Score:   1.6821 | Time: 417.17µs
2026-11-14T19:12:37.266930Z  INFO Epoch 3    | Score:   1.6821 | Time: 511.21µs
2026-11-14T19:12:37.267010Z  INFO Epoch 4    | Score:   1.6821 | Time: 576.83µs
2026-11-14T19:12:37.267086Z  INFO Epoch 5    | Score:   1.6821 | Time: 638.08µs
2026-11-14T19:12:37.267158Z  INFO Epoch 6    | Score:   1.6821 | Time: 696.42µs
2026-11-14T19:12:37.267228Z  INFO Epoch 7    | Score:   1.6821 | Time: 756.29µs
2026-11-14T19:12:37.267299Z  INFO Epoch 8    | Score:   1.6821 | Time: 814.92µs
2026-11-14T19:12:37.267380Z  INFO Epoch 9    | Score:   1.6821 | Time: 882.63µs
2026-11-14T19:12:37.267460Z  INFO Epoch 10   | Score:   1.6821 | Time: 952.00µs
2026-11-14T19:12:37.267543Z  INFO Epoch 11   | Score:   1.6821 | Time: 1.02ms
2026-11-14T19:12:37.267688Z  INFO Epoch 12   | Score: 

### Metrics

Using the `rd.MetricCollector` we can aggregate the metrics for the whole engine's run and visualize the metrics we want to track here once the engine has finished. In this case we only want to see our three expressions, but we could also see any `rd.Metric` that the engine tracks - refer to the docs for more details.

In [8]:
# collector.plot("species.count", "species.threshold", "error")
collector.plot_titled(
    "Species Count vs Rolling 25gen Mean Species Count",
    ["species_count_mean", "species.count"],
)

In [6]:
df = collector.to_polars()
df.filter(pl.col("name") == "species.count").tail(20)
# df

name,last,sum,mean,stddev,var,skew,min,max,count,time_sum,time_mean,time_stddev,time_min,time_max,time_var,generation,update_count,tags
str,f64,f64,f64,f64,f64,f64,f64,f64,i64,duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],duration[μs],i64,i64,list[str]
"""species.count""",3.0,996.0,4.065306,1.076871,1.159652,26.191385,1.0,8.0,245,null,null,null,null,null,null,244,1,"[""species"", ""statistic""]"
"""species.count""",3.0,999.0,4.060976,1.076816,1.159532,26.276598,1.0,8.0,246,null,null,null,null,null,null,245,1,"[""species"", ""statistic""]"
"""species.count""",3.0,1002.0,4.05668,1.076743,1.159376,26.370661,1.0,8.0,247,null,null,null,null,null,null,246,1,"[""species"", ""statistic""]"
"""species.count""",3.0,1005.0,4.052419,1.076654,1.159184,26.473536,1.0,8.0,248,null,null,null,null,null,null,247,1,"[""species"", ""statistic""]"
"""species.count""",3.0,1008.0,4.048193,1.076549,1.158958,26.585039,1.0,8.0,249,null,null,null,null,null,null,248,1,"[""species"", ""statistic""]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""species.count""",3.0,1041.0,4.003846,1.074442,1.154425,28.371744,1.0,8.0,260,null,null,null,null,null,null,259,1,"[""species"", ""statistic""]"
"""species.count""",3.0,1044.0,4.0,1.074172,1.153846,28.585184,1.0,8.0,261,null,null,null,null,null,null,260,1,"[""species"", ""statistic""]"
"""species.count""",3.0,1047.0,3.996183,1.073891,1.153242,28.807312,1.0,8.0,262,null,null,null,null,null,null,261,1,"[""species"", ""statistic""]"
